In [1]:
from scipy.io import loadmat

ROOT_DIR = "/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K"

mat = loadmat(f"{ROOT_DIR}/traindata.mat")

print(mat.keys())

for k in mat.keys():
    if not k.startswith("__"):
        print("\nKEY:", k)
        print("TYPE:", type(mat[k]))
        print("SHAPE:", mat[k].shape)

        try:
            print("FIRST ENTRY:")
            print(mat[k][0])
        except:
            pass

dict_keys(['__header__', '__version__', '__globals__', 'traindata'])

KEY: traindata
TYPE: <class 'numpy.ndarray'>
SHAPE: (1, 2000)
FIRST ENTRY:
[(array(['train/1009_2.png'], dtype='<U16'), array(['YOU'], dtype='<U3'), array([[array(['YOU'], dtype='<U3'), array(['1200'], dtype='<U4'),
         array(['1300'], dtype='<U4'), array(['135'], dtype='<U3'),
         array(['149'], dtype='<U3'), array(['324'], dtype='<U3'),
         array(['365'], dtype='<U3'), array(['54'], dtype='<U2'),
         array(['9'], dtype='<U1'), array(['ABUSE'], dtype='<U5'),
         array(['AD'], dtype='<U2'), array(['ADAMS'], dtype='<U5'),
         array(['BABY'], dtype='<U4'), array(['BEFORE'], dtype='<U6'),
         array(['BLUSH'], dtype='<U5'), array(['BRIDGE'], dtype='<U6'),
         array(['BUDDY'], dtype='<U5'), array(['BUKAN'], dtype='<U5'),
         array(['CARGAINT'], dtype='<U8'), array(['CASTLE'], dtype='<U6'),
         array(['CHAPEL'], dtype='<U6'), array(['CHUTNEYS'], dtype='<U8'),
         array

In [2]:
# ============================================================
# HVLT FOR IIIT5k HANDWRITTEN WORD RECOGNITION
# FULL SINGLE-CELL TRAINING NOTEBOOK (FIXED VERSION)
# ============================================================

# ============================================================
# INSTALLS (RUN ONCE IF NEEDED)
# ============================================================

# !pip install timm transformers jiwer opencv-python pillow tqdm scikit-learn

# ============================================================
# IMPORTS
# ============================================================

import os
import cv2
import time
import random
import unicodedata
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.amp import autocast, GradScaler
import timm
from transformers import XLMRobertaModel
from jiwer import wer
from scipy.io import loadmat

# ============================================================
# CONFIG
# ============================================================

ROOT_DIR = "/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K"

TRAIN_MAT = os.path.join(ROOT_DIR, "traindata.mat")
TEST_MAT  = os.path.join(ROOT_DIR, "testdata.mat")

IMG_HEIGHT = 32
IMG_WIDTH = 192

BATCH_SIZE = 32

LR = 5e-5

MAX_EPOCHS = 20

MAX_SEQ_LEN = 40

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NUM_WORKERS = 4

SEED = 42

USE_AMP = True

# ============================================================
# SEED
# ============================================================

def set_seed(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# ENGLISH VOCABULARY (IIIT5K)
# ============================================================

CHARS = (
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
    ".,!?()[]{}:;-_'/\\&%$#@+*= "
)

SPECIAL_TOKENS = [
    "<PAD>",
    "<SOS>",
    "<EOS>",
    "<UNK>",
]

ALL_TOKENS = SPECIAL_TOKENS + list(CHARS)

char2idx = {
    c: i for i, c in enumerate(ALL_TOKENS)
}

idx2char = {
    i: c for c, i in char2idx.items()
}

VOCAB_SIZE = len(ALL_TOKENS)

PAD_IDX = char2idx["<PAD>"]
SOS_IDX = char2idx["<SOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

print("VOCAB SIZE:", VOCAB_SIZE)
# ============================================================
# LOAD IIIT5K TRAIN DATA
# ============================================================

samples = []

mat = loadmat(TRAIN_MAT)

records = mat["traindata"][0]

for record in records:

    try:

        img_rel = str(record[0][0])

        text = str(record[1][0])

        img_path = os.path.join(
            ROOT_DIR,
            img_rel
        )

        if os.path.exists(img_path):

            samples.append(
                (
                    img_path,
                    unicodedata.normalize(
                        "NFC",
                        text.strip()
                    )
                )
            )

    except Exception:
        continue

print("TOTAL TRAIN SAMPLES:", len(samples))

# ============================================================
# HELPER
# ============================================================

def extract_string(x):

    while isinstance(x, np.ndarray):

        if x.size == 0:
            return ""

        x = x[0]

    return str(x)


# ============================================================
# LOAD IIIT5K TRAIN DATA
# ============================================================

samples = []

train_mat = loadmat(TRAIN_MAT)

records = train_mat["traindata"][0]

for record in records:

    try:

        img_rel = extract_string(record[0])

        text = extract_string(record[1])

        img_path = os.path.join(
            ROOT_DIR,
            img_rel
        )

        if os.path.exists(img_path):

            samples.append(
                (
                    img_path,
                    unicodedata.normalize(
                        "NFC",
                        text.strip()
                    )
                )
            )

    except Exception as e:

        continue

print("TOTAL TRAIN SAMPLES:", len(samples))


# ============================================================
# LOAD IIIT5K TEST DATA
# ============================================================

test_samples = []

test_mat = loadmat(TEST_MAT)

records = test_mat["testdata"][0]

for record in records:

    try:

        img_rel = extract_string(record[0])

        text = extract_string(record[1])

        img_path = os.path.join(
            ROOT_DIR,
            img_rel
        )

        if os.path.exists(img_path):

            test_samples.append(
                (
                    img_path,
                    unicodedata.normalize(
                        "NFC",
                        text.strip()
                    )
                )
            )

    except Exception as e:

        continue

print("TOTAL TEST SAMPLES:", len(test_samples))


# ============================================================
# DEBUG PRINTS
# ============================================================

print("\nFIRST 5 TRAIN SAMPLES")

for i in range(min(5, len(samples))):

    print(samples[i])

print("\nFIRST 5 TEST SAMPLES")

for i in range(min(5, len(test_samples))):

    print(test_samples[i])


# ============================================================
# SPLIT TRAIN -> TRAIN + VAL
# ============================================================

train_samples, val_samples = train_test_split(
    samples,
    test_size=0.10,
    random_state=SEED,
)

print("TRAIN:", len(train_samples))
print("VAL:", len(val_samples))
print("TEST:", len(test_samples))

# ============================================================
# IMAGE PREPROCESS
# ============================================================

def preprocess_image(img_path):

    img = cv2.imread(img_path)

    if img is None:
        raise ValueError(f"Cannot read image: {img_path}")

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    h, w = img.shape[:2]

    if h <= 0 or w <= 0:
        raise ValueError(f"Invalid image size: {img_path}")

    scale = IMG_HEIGHT / h

    new_w = max(1, int(w * scale))

    img = cv2.resize(
        img,
        (new_w, IMG_HEIGHT)
    )

    if new_w < IMG_WIDTH:

        pad = np.ones(
            (
                IMG_HEIGHT,
                IMG_WIDTH - new_w,
                3
            ),
            dtype=np.uint8
        ) * 255

        img = np.concatenate(
            [img, pad],
            axis=1
        )

    else:

        img = cv2.resize(
            img,
            (IMG_WIDTH, IMG_HEIGHT)
        )

    img = img.astype(np.float32) / 255.0

    img = (img - 0.5) / 0.5

    img = np.transpose(
        img,
        (2, 0, 1)
    )

    return torch.tensor(
        img,
        dtype=torch.float32
    )

# ============================================================
# TEXT ENCODING
# ============================================================

def encode_text(text):

    tokens = [SOS_IDX]

    for ch in text:

        tokens.append(
            char2idx.get(ch, UNK_IDX)
        )

    tokens.append(EOS_IDX)

    if len(tokens) < MAX_SEQ_LEN:

        tokens += [PAD_IDX] * (
            MAX_SEQ_LEN - len(tokens)
        )

    else:

        tokens = tokens[:MAX_SEQ_LEN]

        tokens[-1] = EOS_IDX

    return torch.tensor(
        tokens,
        dtype=torch.long
    )

# ============================================================
# TEXT DECODING
# ============================================================

def decode_tokens(tokens):

    chars = []

    for t in tokens:

        t = int(t)

        if t == EOS_IDX:
            break

        if t in [PAD_IDX, SOS_IDX]:
            continue

        chars.append(
            idx2char.get(t, "")
        )

    return "".join(chars)

# ============================================================
# DATASET
# ============================================================

class IIIT5KWordDataset(Dataset):

    def __init__(self, samples):

        self.samples = samples

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        img_path, text = self.samples[idx]

        image = preprocess_image(img_path)

        tokens = encode_text(text)

        return image, tokens, text

# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    IIIT5KWordDataset(train_samples),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    IIIT5KWordDataset(val_samples),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    IIIT5KWordDataset(test_samples),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

# ============================================================
# POSITIONAL BRIDGE
# ============================================================

class PositionalBridge(nn.Module):

    def __init__(
        self,
        in_channels=1024,
        d_model=768,
        vis_seq_len=256,
    ):
        super().__init__()

        self.pool = nn.AdaptiveAvgPool2d(
            (1, vis_seq_len)
        )

        self.proj = nn.Linear(
            in_channels,
            d_model,
        )

        self.pos_embed = nn.Parameter(
            torch.randn(
                1,
                vis_seq_len,
                d_model,
            ) * 0.02
        )

    def forward(self, x):

        # x = (B,H,W,C)

        B, H, W, C = x.shape

        # -> (B,C,H,W)
        x = x.permute(0, 3, 1, 2)

        # -> (B,C,1,T)
        x = self.pool(x)

        # -> (B,C,T)
        x = x.squeeze(2)

        # -> (B,T,C)
        x = x.permute(0, 2, 1)

        # 1024 -> 768
        x = self.proj(x)

        x = x + self.pos_embed

        return x

# ============================================================
# DECODER
# ============================================================

class IIIT5KDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_layers=6,
    ):
        super().__init__()

        self.token_embed = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_IDX,
        )

        self.pos_embed = nn.Embedding(
            MAX_SEQ_LEN,
            d_model,
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=3072,
            dropout=0.1,
            batch_first=True,
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=n_layers,
        )

        self.output_proj = nn.Linear(
            d_model,
            vocab_size,
        )

        print("Loading XLM-RoBERTa...")

        try:

            xlm = XLMRobertaModel.from_pretrained(
                "xlm-roberta-base"
            )

            emb = xlm.embeddings.word_embeddings.weight

            n = min(
                emb.shape[0],
                vocab_size,
            )

            self.token_embed.weight.data[:n] = emb[:n]

            del xlm

            print("Loaded multilingual weights.")

        except Exception as e:

            print("Pretrained load failed:", e)

    def forward(
        self,
        memory,
        tgt_tokens,
    ):

        B, T = tgt_tokens.shape

        pos = torch.arange(
            T,
            device=tgt_tokens.device,
        ).unsqueeze(0).expand(B, -1)

        x = (
            self.token_embed(tgt_tokens)
            + self.pos_embed(pos)
        )

        tgt_mask = torch.triu(
            torch.ones(
                T,
                T,
                device=tgt_tokens.device,
            ),
            diagonal=1,
        ).bool()

        tgt_key_padding_mask = (
            tgt_tokens == PAD_IDX
        )

        out = self.decoder(
            tgt=x,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
        )

        logits = self.output_proj(out)

        return logits

    @torch.no_grad()
    def greedy_decode(
        self,
        memory,
        max_len=MAX_SEQ_LEN,
    ):

        B = memory.shape[0]

        generated = torch.full(
            (B, 1),
            SOS_IDX,
            device=memory.device,
            dtype=torch.long,
        )

        finished = torch.zeros(
            B,
            dtype=torch.bool,
            device=memory.device,
        )

        for _ in range(max_len):

            logits = self.forward(
                memory,
                generated,
            )

            next_token = logits[:, -1].argmax(dim=-1)

            next_token = next_token.masked_fill(
                finished,
                PAD_IDX,
            )

            generated = torch.cat(
                [
                    generated,
                    next_token.unsqueeze(1),
                ],
                dim=1,
            )

            finished |= (
                next_token == EOS_IDX
            )

            if finished.all():
                break

        return generated[:, 1:]

# ============================================================
# HVLT MODEL
# ============================================================

class HVLT(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(32, 192),
            strict_img_size=False,
        )

        self.bridge = PositionalBridge(
            in_channels=1024,
            d_model=768,
            vis_seq_len=256,
        )

        self.decoder = IIIT5KDecoder(
            vocab_size=VOCAB_SIZE
        )

    def forward(
        self,
        images,
        tgt_tokens,
    ):

        # timm already returns (B,H,W,C)
        feats = self.encoder(images)[-1]

        memory = self.bridge(feats)

        logits = self.decoder(
            memory,
            tgt_tokens,
        )

        return logits

    @torch.no_grad()
    def predict(self, images):

        feats = self.encoder(images)[-1]

        memory = self.bridge(feats)

        preds = self.decoder.greedy_decode(memory)

        return preds
    
# ============================================================
# LOSS
# ============================================================

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

# ============================================================
# METRICS
# ============================================================

def char_accuracy(preds, labels):

    correct = 0

    total = 0

    for p, l in zip(preds, labels):

        n = min(len(p), len(l))

        for i in range(n):

            if p[i] == l[i]:
                correct += 1

        total += max(len(p), len(l))

    return 100.0 * correct / max(total, 1)

# ============================================================
# MODEL
# ============================================================

model = HVLT().to(DEVICE)

optimizer = Adam(
    model.parameters(),
    lr=LR,
)

scaler = GradScaler(
    "cuda",
    enabled=USE_AMP
)

print(
    "TOTAL PARAMS:",
    sum(
        p.numel()
        for p in model.parameters()
    ) / 1e6,
    "M"
)

# ============================================================
# TRAINING LOOP
# ============================================================

best_wer = 9999

for epoch in range(1, MAX_EPOCHS + 1):

    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    train_loss = 0

    train_preds = []

    train_labels = []

    pbar = tqdm(train_loader)

    for images, targets, labels in pbar:

        images = images.to(DEVICE)

        targets = targets.to(DEVICE)

        decoder_input = targets[:, :-1]

        decoder_target = targets[:, 1:]

        optimizer.zero_grad()

        with autocast(
            "cuda",
            enabled=USE_AMP
        ):

            logits = model(
                images,
                decoder_input,
            )

            loss = criterion(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                decoder_target.reshape(-1),
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0,
        )

        scaler.step(optimizer)

        scaler.update()

        train_loss += loss.item()

        with torch.no_grad():

            pred_ids = logits.argmax(dim=-1)

            preds = [
                decode_tokens(x)
                for x in pred_ids
            ]

        train_preds.extend(preds)

        train_labels.extend(labels)

        pbar.set_description(
            f"Train Loss: {loss.item():.4f}"
        )

    train_cer = char_accuracy(
        train_preds,
        train_labels,
    )

    train_wer = wer(
        train_labels,
        train_preds,
    ) * 100

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss = 0

    val_preds = []

    val_labels = []

    with torch.no_grad():

        pbar = tqdm(val_loader)

        for images, targets, labels in pbar:

            images = images.to(DEVICE)

            targets = targets.to(DEVICE)

            decoder_input = targets[:, :-1]

            decoder_target = targets[:, 1:]

            logits = model(
                images,
                decoder_input,
            )

            loss = criterion(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                decoder_target.reshape(-1),
            )

            val_loss += loss.item()

            pred_ids = model.predict(images)

            preds = [
                decode_tokens(x)
                for x in pred_ids
            ]

            val_preds.extend(preds)

            val_labels.extend(labels)

    val_cer = char_accuracy(
        val_preds,
        val_labels,
    )

    val_wer = wer(
        val_labels,
        val_preds,
    ) * 100

    # ========================================================
    # LOGGING
    # ========================================================

    print("\n")
    print("=" * 60)

    print(f"EPOCH {epoch}")

    print(
        f"TRAIN LOSS: {train_loss / len(train_loader):.4f}"
    )

    print(
        f"VAL LOSS: {val_loss / len(val_loader):.4f}"
    )

    print(
        f"TRAIN CAR: {train_cer:.2f}%"
    )

    print(
        f"VAL CAR: {val_cer:.2f}%"
    )

    print(
        f"TRAIN WER: {train_wer:.2f}%"
    )

    print(
        f"VAL WER: {val_wer:.2f}%"
    )

    print("=" * 60)

    # ========================================================
    # SAVE BEST
    # ========================================================

    if val_wer < best_wer:

        best_wer = val_wer

        torch.save(
            {
                "model": model.state_dict(),
                "epoch": epoch,
                "val_wer": val_wer,
            },
            "best_iiit5k_hvlt.pth",
        )

        print("BEST MODEL SAVED.")

# ============================================================
# TEST EVALUATION
# ============================================================

print("\nFINAL TEST EVALUATION")

model.eval()

test_preds = []

test_labels = []

with torch.no_grad():

    for images, targets, labels in tqdm(test_loader):

        images = images.to(DEVICE)

        pred_ids = model.predict(images)

        preds = [
            decode_tokens(x)
            for x in pred_ids
        ]

        test_preds.extend(preds)

        test_labels.extend(labels)

test_cer = char_accuracy(
    test_preds,
    test_labels,
)

test_wer = wer(
    test_labels,
    test_preds,
) * 100

print("\nTEST RESULTS")

print("TEST CAR:", test_cer)

print("TEST WER:", test_wer)

# ============================================================
# SAMPLE PREDICTIONS
# ============================================================

print("\nSAMPLE PREDICTIONS\n")

for i in range(10):

    print("GT   :", test_labels[i])

    print("PRED :", test_preds[i])

    print("-" * 50)

/home/mca/Shweta/handwritten text detection/sbenv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


VOCAB SIZE: 92
TOTAL TRAIN SAMPLES: 2000
TOTAL TRAIN SAMPLES: 2000
TOTAL TEST SAMPLES: 3000

FIRST 5 TRAIN SAMPLES
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/train/1009_2.png', 'YOU')
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/train/1017_1.png', 'RESCUE')
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/train/1017_2.png', 'MISSION')
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/train/1021_1.png', 'HOME')
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/train/1023_1.png', 'BORDER')

FIRST 5 TEST SAMPLES
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/test/1002_1.png', 'PRIVATE')
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/test/1002_2.png', 'PARKING')
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/test/100

/home/mca/Shweta/handwritten text detection/sbenv/lib/python3.12/site-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at /pytorch/aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)
/home/mca/Shweta/handwritten text detection/sbenv/lib/python3.12/site-packages/timm/layers/interpolate.py:65: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:34

Loading XLM-RoBERTa...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...



EPOCH 19
TRAIN LOSS: 0.2788
VAL LOSS: 1.9097
TRAIN CAR: 87.82%
VAL CAR: 43.25%
TRAIN WER: 27.22%
VAL WER: 82.00%


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:01<00:00,  3.83it/s]




EPOCH 20
TRAIN LOSS: 0.2385
VAL LOSS: 1.9785
TRAIN CAR: 88.89%
VAL CAR: 40.71%
TRAIN WER: 23.11%
VAL WER: 83.50%

FINAL TEST EVALUATION


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 94/94 [00:15<00:00,  6.07it/s]


TEST RESULTS
TEST CAR: 36.79350520575112
TEST WER: 81.63333333333334

SAMPLE PREDICTIONS

GT   : PRIVATE
PRED : PRAYERE
--------------------------------------------------
GT   : PARKING
PRED : RARKINE
--------------------------------------------------
GT   : SALUTES
PRED : SPITES
--------------------------------------------------
GT   : DOLCE
PRED : DOGE
--------------------------------------------------
GT   : GABBANA
PRED : CARSANAR
--------------------------------------------------
GT   : REGENCY
PRED : TRERKANG
--------------------------------------------------
GT   : STATE
PRED : SELAR
--------------------------------------------------
GT   : BANK
PRED : JELAN
--------------------------------------------------
GT   : OF
PRED : 0LE
--------------------------------------------------
GT   : INDIA
PRED : NENE
--------------------------------------------------


## TPS+ResNet+GCNN+BiLSTM+Attention

In [3]:
# ============================================================
# ICDAR HWD 2024 — WORD LEVEL OCR (STABLE RESNET + GATED CNN)
# Production-Ready OCR Pipeline with AdamW, AMP & Precise Metric Averaging
# ============================================================

import os
import random
import numpy as np
import cv2
from PIL import Image
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from Levenshtein import distance

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from tqdm import tqdm
import matplotlib.pyplot as plt

# ============================================================
# DEVICE CONFIGURATION
# ============================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ============================================================
# CONFIGURATION CONSTANTS
# ============================================================

SAVE_DIR = "/home/mca/Shweta/handwritten text detection/saved_models/iiit5k"
ROOT_DIR = "/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K"
TRAIN_MAT = os.path.join(ROOT_DIR, "traindata.mat")
TEST_MAT  = os.path.join(ROOT_DIR, "testdata.mat")
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 32
NUM_EPOCHS = 80
LEARNING_RATE = 1e-3  
NUM_WORKERS = 4
MAX_LABEL_LEN = 30
IMG_H = 32
IMG_W = 100
VAL_SPLIT = 0.1
PATIENCE = 20         
HIDDEN_SIZE = 256

# ============================================================
# VOCABULARY SETUP
# ============================================================

CHARS = ' !"#$%&\'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\\]^_`abcdefghijklmnopqrstuvwxyz{|}~'

GO_IDX  = 0
EOS_IDX = 1
PAD_IDX = 2
UNK_IDX = 3

CHAR_TO_IDX = {c: i + 4 for i, c in enumerate(CHARS)}
IDX_TO_CHAR = {v: k for k, v in CHAR_TO_IDX.items()}

IDX_TO_CHAR.update({
    GO_IDX: "<GO>",
    EOS_IDX: "<EOS>",
    PAD_IDX: "<PAD>",
    UNK_IDX: "<UNK>"
})

NUM_CLASSES = len(CHARS) + 4
print("NUM_CLASSES =", NUM_CLASSES)

# ============================================================
# LABEL PROCESSING UTILITIES
# ============================================================

def encode_label(text, max_len):
    tokens = [GO_IDX]
    for c in text[:max_len]:
        tokens.append(CHAR_TO_IDX.get(c, UNK_IDX))
    tokens.append(EOS_IDX)
    while len(tokens) < max_len + 2:
        tokens.append(PAD_IDX)
    return torch.tensor(tokens[:max_len+2], dtype=torch.long)

def decode_label(tensor):
    chars = []
    for idx in tensor.tolist():
        if idx in [EOS_IDX, PAD_IDX]:
            break
        if idx in [GO_IDX, UNK_IDX]:
            continue
        chars.append(IDX_TO_CHAR.get(idx, ""))
    return "".join(chars)

# ============================================================
# IMAGE TRANSFORMS & DATA LOADING
# ============================================================

def resize_keep_ratio(img):
    img = np.array(img)
    h, w = img.shape[:2]
    ratio = IMG_H / h
    new_w = int(w * ratio)

    img = cv2.resize(img, (new_w, IMG_H))

    if new_w < IMG_W:
        pad = np.ones((IMG_H, IMG_W - new_w, 3), dtype=np.uint8) * 255
        img = np.concatenate([img, pad], axis=1)
    else:
        img = cv2.resize(img, (IMG_W, IMG_H))
    return Image.fromarray(img)

class IIIT5KDataset(Dataset):
    def __init__(self, samples, max_label_len=25, augment=False):
        self.samples = samples
        self.max_label_len = max_label_len
        self.augment = augment

        if self.augment:
            self.transform = transforms.Compose([
                transforms.Grayscale(1),
                transforms.RandomAffine(
                    degrees=2,
                    translate=(0.02, 0.02),
                    scale=(0.95, 1.05)
                ),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(),
                transforms.Normalize([0.5], [0.5])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Grayscale(1),
                transforms.ToTensor(),
                transforms.Normalize([0.5], [0.5])
            ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        image = resize_keep_ratio(image)
        image = self.transform(image)
        encoded = encode_label(label, self.max_label_len)
        return image, encoded, label

def load_iiit5k(mat_path):
    samples = []
    mat = loadmat(mat_path)
    key = "traindata" if "traindata" in mat else "testdata"
    records = mat[key][0]
    for record in records:
        try:
            img_rel = str(record[0][0])
            text = str(record[1][0]).strip()
            img_path = os.path.join(ROOT_DIR, img_rel)
            if os.path.exists(img_path):
                samples.append((img_path, text))
        except:
            pass
    return samples

# Read and generate Data splits
train_all = load_iiit5k(TRAIN_MAT)
test_samples = load_iiit5k(TEST_MAT)
train_samples, val_samples = train_test_split(train_all, test_size=VAL_SPLIT, random_state=42)

for i in range(20):
    print(train_samples[i])

print(f"Train samples count: {len(train_samples)}")
print(f"Val samples count:   {len(val_samples)}")
print(f"Test samples count:  {len(test_samples)}")

train_ds = IIIT5KDataset(train_samples, MAX_LABEL_LEN, augment=True)
val_ds   = IIIT5KDataset(val_samples, MAX_LABEL_LEN, augment=False)
test_ds  = IIIT5KDataset(test_samples, MAX_LABEL_LEN, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# ============================================================
# COMPACT RESNET + GATED CNN FEATURE EXTRACTOR
# ============================================================

class GatedCNNLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.conv_f = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.conv_g = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        f = self.conv_f(x)
        g = torch.sigmoid(self.conv_g(x))
        return self.bn(f * g)

class ResNet_GatedCNN_FeatureExtractor(nn.Module):
    """
    Input: [B, 1, 32, 100] -> Sequence Output Size: [B, 512, 4, 25]
    """
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.gate1 = GatedCNNLayer(64, 128)
        self.gate2 = GatedCNNLayer(128, 128)
        self.gate3 = GatedCNNLayer(128, 256)
        self.gate4 = GatedCNNLayer(256, 256)
        self.gate5 = GatedCNNLayer(256, 512)
        self.gate6 = GatedCNNLayer(512, 512)
        
        self.pool2d = nn.MaxPool2d(2, 2)                     
        self.pool_w = nn.MaxPool2d((2, 1), (2, 1))           

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool2d(F.relu(self.bn2(self.conv2(x))))     # Shape: [B, 64, 16, 50]
        
        x = self.gate1(x)
        x = self.pool2d(self.gate2(x))                      # Shape: [B, 128, 8, 25]
        
        x = self.gate3(x)
        x = self.pool_w(self.gate4(x))                      # Shape: [B, 256, 4, 25]
        
        x = self.gate5(x)
        x = self.gate6(x)                                   # Shape: [B, 512, 4, 25]
        return x

# ============================================================
# SEQUENTIAL MODELLING COMPONENTS
# ============================================================

class BidirectionalLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, bidirectional=True, batch_first=True)
        self.linear = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):
        x, _ = self.rnn(x)
        return self.linear(x)

class AttentionCell(nn.Module):
    def __init__(self, input_size, hidden_size, num_embeddings):
        super().__init__()
        self.i2h = nn.Linear(input_size, hidden_size, bias=False)
        self.h2h = nn.Linear(hidden_size, hidden_size)
        self.score = nn.Linear(hidden_size, 1, bias=False)
        self.rnn = nn.LSTMCell(input_size + num_embeddings, hidden_size)

    def forward(self, prev_hidden, batch_H, char_onehots):
        batch_H_proj = self.i2h(batch_H)
        prev_hidden_proj = self.h2h(prev_hidden[0]).unsqueeze(1)
        e = self.score(torch.tanh(batch_H_proj + prev_hidden_proj))
        alpha = F.softmax(e, dim=1)
        context = torch.bmm(alpha.permute(0, 2, 1), batch_H).squeeze(1)
        concat_context = torch.cat([context, char_onehots], dim=1)
        cur_hidden = self.rnn(concat_context, prev_hidden)
        return cur_hidden, alpha

class Attention(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.attention_cell = AttentionCell(input_size, hidden_size, num_classes)
        self.hidden_size = hidden_size
        self.num_classes = num_classes
        self.generator = nn.Linear(hidden_size, num_classes)

    def _char_to_onehot(self, input_char):
        one_hot = torch.zeros(input_char.size(0), self.num_classes, device=input_char.device)
        one_hot.scatter_(1, input_char.unsqueeze(1), 1)
        return one_hot

    def forward(self, batch_H, text=None, is_train=True, batch_max_length=25):
        batch_size = batch_H.size(0)
        num_steps = batch_max_length + 1
        hidden = (
            torch.zeros(batch_size, self.hidden_size, device=batch_H.device),
            torch.zeros(batch_size, self.hidden_size, device=batch_H.device)
        )
        output_hiddens = torch.zeros(batch_size, num_steps, self.hidden_size, device=batch_H.device)

        if is_train:
            for i in range(num_steps):
                char_onehots = self._char_to_onehot(text[:, i])
                hidden, alpha = self.attention_cell(hidden, batch_H, char_onehots)
                output_hiddens[:, i, :] = hidden[0]
            probs = self.generator(output_hiddens)
        else:
            targets = torch.LongTensor(batch_size).fill_(GO_IDX).to(batch_H.device)
            probs = torch.zeros(batch_size, num_steps, self.num_classes, device=batch_H.device)
            for i in range(num_steps):
                char_onehots = self._char_to_onehot(targets)
                hidden, alpha = self.attention_cell(hidden, batch_H, char_onehots)
                probs_step = self.generator(hidden[0])
                probs[:, i, :] = probs_step
                _, next_input = probs_step.max(1)
                targets = next_input
        return probs

# ============================================================
# INTEGRATED OCR SYSTEM MODULE
# ============================================================

class OCRModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.Transformation = nn.Identity()
        self.FeatureExtraction = ResNet_GatedCNN_FeatureExtractor()
        self.AdaptiveAvgPool = nn.AdaptiveAvgPool2d((None, 1))
        
        self.SequenceModeling = nn.Sequential(
            BidirectionalLSTM(512, HIDDEN_SIZE, HIDDEN_SIZE),
            nn.Dropout(0.3),
            BidirectionalLSTM(HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE),
            nn.Dropout(0.3)
        )
        self.Prediction = Attention(HIDDEN_SIZE, HIDDEN_SIZE, NUM_CLASSES)

    def forward(self, images, text=None, is_train=True):
        rectified_features = self.Transformation(images)
        visual_feature = self.FeatureExtraction(rectified_features)
        visual_feature = self.AdaptiveAvgPool(visual_feature.permute(0, 3, 1, 2)).squeeze(3)
        contextual_feature = self.SequenceModeling(visual_feature)
        prediction = self.Prediction(contextual_feature.contiguous(), text, is_train, MAX_LABEL_LEN)
        return prediction

# Instantiate model architecture
model = OCRModel().to(device)
print("Parameters =", sum(p.numel() for p in model.parameters() if p.requires_grad))

# ============================================================
# OPTIMIZERS, COMPILER SCALERS & LOSS INFRASTRUCTURE
# ============================================================

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# Switched to AdamW to correctly handle weight decay on structural layers
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# ============================================================
# METRICS EVALUATION LOGIC
# ============================================================

def compute_metrics(logits, labels):
    preds = logits.argmax(dim=-1)
    pred_texts = []
    gt_texts = []

    for pred_seq, gt_seq in zip(preds, labels):
        pred_texts.append(decode_label(pred_seq))
        gt_texts.append(decode_label(gt_seq))

    correct = sum(p == g for p, g in zip(pred_texts, gt_texts))
    word_acc = correct / len(gt_texts)

    total_chars = 0
    total_edits = 0
    for p, g in zip(pred_texts, gt_texts):
        total_chars += len(g)
        total_edits += distance(p, g)

    char_acc = max(0.0, 1 - (total_edits / max(total_chars, 1)))
    return word_acc, char_acc

# ============================================================
# VALIDATION EXECUTION INFRASTRUCTURE
# ============================================================

def run_validation():
    model.eval()
    total_loss = 0.0
    total_wacc = 0.0
    total_cacc = 0.0
    total_samples = 0

    with torch.no_grad():
        for images, labels, _ in val_loader:
            batch_size = images.size(0)
            images = images.to(device)
            labels = labels.to(device)
            target = labels[:, 1:]

            # Teacher Forcing tracking mode
            with torch.cuda.amp.autocast(enabled=use_amp):
                loss_logits = model(images, labels[:, :-1], is_train=True)
                loss = criterion(loss_logits.reshape(-1, NUM_CLASSES), target.reshape(-1))
            total_loss += loss.item()

            # Autoregressive decoding logic execution
            inference_logits = model(images, text=None, is_train=False)
            wacc, cacc = compute_metrics(inference_logits.cpu(), labels.cpu())

            # Weighing accumulations per batch sample size to eliminate partial-batch evaluation bias
            total_wacc += wacc * batch_size
            total_cacc += cacc * batch_size
            total_samples += batch_size

    return (total_loss / len(val_loader), total_wacc / total_samples, total_cacc / total_samples)
def run_test():
    model.eval()

    total_wacc = 0.0
    total_cacc = 0.0
    total_samples = 0

    with torch.no_grad():
        for images, labels, _ in test_loader:

            batch_size = images.size(0)

            images = images.to(device)
            labels = labels.to(device)

            logits = model(images, text=None, is_train=False)

            wacc, cacc = compute_metrics(
                logits.cpu(),
                labels.cpu()
            )

            total_wacc += wacc * batch_size
            total_cacc += cacc * batch_size
            total_samples += batch_size

    return (
        total_wacc / total_samples,
        total_cacc / total_samples
    )
    
# ============================================================
# MAIN TRAINING PIPELINE ENGINE
# ============================================================

best_val_acc = 0
patience_counter = 0

history = {"train_loss": [], "val_loss": [], "val_wacc": []}

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0
    pbar = tqdm(train_loader)

    for images, labels, _ in pbar:
        images = images.to(device)
        labels = labels.to(device)
        target = labels[:, 1:]

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(images, labels[:, :-1], is_train=True)
            loss = criterion(logits.reshape(-1, NUM_CLASSES), target.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pbar.set_description(f"Epoch {epoch+1}/{NUM_EPOCHS}")
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    scheduler.step()
    train_loss = running_loss / len(train_loader)
    val_loss, val_wacc, val_cacc = run_validation()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_wacc"].append(val_wacc)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val WAcc   : {val_wacc*100:.2f}%")
    print(f"Val CAcc   : {val_cacc*100:.2f}%")

    torch.save(model.state_dict(), os.path.join(SAVE_DIR, "last_iiit5k_2.pth"))

    if val_wacc > best_val_acc:
        best_val_acc = val_wacc
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_iiit5k_2.pth"))
        print("--- BEST MODEL HIGHLIGHT TRACKED & SAVED ---")
    else:
        patience_counter += 1
        print(f"Early Stopping Status Count: {patience_counter} out of limit of {PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n[Termination Alert] System triggered early stopping limit reached at Epoch {epoch+1}!")
        break

print(f"\nPipeline Run Completed. Maximum Word Accuracy Checked: {best_val_acc*100:.2f}%")
print("\nLoading best model...")

best_model_path = os.path.join(SAVE_DIR, "best.pth")

model.load_state_dict(
    torch.load(best_model_path, map_location=device)
)

test_wacc, test_cacc = run_test()

print("\n===== OFFICIAL TEST RESULTS =====")
print(f"Test Word Accuracy : {test_wacc*100:.2f}%")
print(f"Test Char Accuracy : {test_cacc*100:.2f}%")

# ============================================================
# PERFORMANCE CURVE PLOTS
# ============================================================

plt.figure(figsize=(10, 5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(SAVE_DIR, "loss_curve.png"))
plt.show()

Device : cuda
GPU    : NVIDIA RTX 5000 Ada Generation
VRAM   : 33.8 GB
NUM_CLASSES = 99


/tmp/ipykernel_741007/2781627414.py:359: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/train/927_2.png', 'BLAZE')
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/train/2429_1.png', 'SANDY')
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/train/2055_32.png', 'NUMBERS')
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/train/2624_3.png', '2')
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/train/2311_5.png', 'GENOAS')
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/train/2114_6.png', 'FLICKER')
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/train/632_1.png', 'POWER')
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/train/529_15.png', 'VIRGINIA')
('/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K/train/5073_7.png', 'AMERICAN')
('/h

  0%|                                                                                                                                    | 0/57 [00:00<?, ?it/s]/tmp/ipykernel_741007/2781627414.py:472: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
Epoch 1/80: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 57/57 [00:03<00:00, 17.40it/s, loss=3.1600]
/tmp/ipykernel_741007/2781627414.py:405: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):



Epoch 1
Train Loss : 3.1688
Val Loss   : 3.0826
Val WAcc   : 0.00%
Val CAcc   : 11.16%
Early Stopping Status Count: 1 out of limit of 20


...

Epoch 79/80: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 57/57 [00:03<00:00, 14.76it/s, loss=0.7809]



Epoch 79
Train Loss : 0.5206
Val Loss   : 1.1446
Val WAcc   : 27.00%
Val CAcc   : 59.80%
Early Stopping Status Count: 7 out of limit of 20


Epoch 80/80: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 57/57 [00:03<00:00, 14.77it/s, loss=0.3566]



Epoch 80
Train Loss : 0.5096
Val Loss   : 1.1578
Val WAcc   : 28.50%
Val CAcc   : 59.67%
--- BEST MODEL HIGHLIGHT TRACKED & SAVED ---

Pipeline Run Completed. Maximum Word Accuracy Checked: 28.50%

Loading best model...


FileNotFoundError: [Errno 2] No such file or directory: '/home/mca/Shweta/handwritten text detection/saved_models/iiit5k/best.pth'

## CNNBACKBONE+BILSTM+LINEAR PROJ+CTC loss (train) / greedy decode (inference)

In [4]:
# ============================================================
# CRNN + CTC PIPELINE FOR IIIT5K WORD RECOGNITION
# Architecture: CNN Backbone → BiLSTM → CTC Loss
# Designed for small datasets (~2000 training images)
# ============================================================

# ============================================================
# INSTALLS (run once if needed)
# pip install timm opencv-python pillow tqdm scikit-learn
#             python-Levenshtein torch torchvision
# ============================================================

import os
import random
import unicodedata
import numpy as np
import cv2
from PIL import Image
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from Levenshtein import distance as levenshtein_distance

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torch.cuda.amp import GradScaler, autocast

from tqdm import tqdm

# ============================================================
# DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ============================================================
# PATHS & HYPERPARAMETERS
# ============================================================

ROOT_DIR  = "/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K"
TRAIN_MAT = os.path.join(ROOT_DIR, "traindata.mat")
TEST_MAT  = os.path.join(ROOT_DIR, "testdata.mat")
SAVE_DIR  = "/home/mca/Shweta/handwritten text detection/saved_models/crnn_ctc"
os.makedirs(SAVE_DIR, exist_ok=True)

# Image
IMG_H       = 32
IMG_W       = 128        # wider than before for better feature resolution

# Training
BATCH_SIZE  = 16         # smaller batch → more gradient steps on small data
NUM_EPOCHS  = 150
LR          = 3e-4       # stable starting LR for AdamW
WEIGHT_DECAY= 1e-4
VAL_SPLIT   = 0.10
PATIENCE    = 30         # generous patience for small dataset
SEED        = 42
NUM_WORKERS = 4
USE_AMP     = torch.cuda.is_available()

# Model
RNN_HIDDEN  = 256
RNN_LAYERS  = 2

# ============================================================
# SEED
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# VOCABULARY
# ============================================================
# CTC uses a blank token (index 0) — all other chars shift by 1

CHARS = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

BLANK_IDX   = 0
char2idx    = {c: i + 1 for i, c in enumerate(CHARS)}   # 1-indexed
idx2char    = {v: k for k, v in char2idx.items()}
NUM_CLASSES = len(CHARS) + 1                             # +1 for CTC blank

print(f"Vocab size (with blank): {NUM_CLASSES}")

def encode_label(text):
    """Convert string to list of int indices (no blank, no padding)."""
    return [char2idx.get(c, 0) for c in text if c in char2idx]

def decode_ctc(indices):
    """
    Collapse CTC output:
    1. Remove consecutive duplicates
    2. Remove blank (0)
    """
    result = []
    prev = None
    for idx in indices:
        if idx != prev:
            if idx != BLANK_IDX:
                result.append(idx)
            prev = idx
    return "".join(idx2char.get(i, "") for i in result)

# ============================================================
# DATA LOADING
# ============================================================

def extract_string(x):
    while isinstance(x, np.ndarray):
        if x.size == 0:
            return ""
        x = x[0]
    return str(x)

def load_iiit5k(mat_path, key):
    samples = []
    mat = loadmat(mat_path)
    records = mat[key][0]
    for record in records:
        try:
            img_rel = extract_string(record[0])
            text    = extract_string(record[1]).strip()
            text    = unicodedata.normalize("NFC", text)
            img_path = os.path.join(ROOT_DIR, img_rel)
            if os.path.exists(img_path) and len(text) > 0:
                samples.append((img_path, text))
        except Exception:
            continue
    return samples

train_all    = load_iiit5k(TRAIN_MAT, "traindata")
test_samples = load_iiit5k(TEST_MAT,  "testdata")

train_samples, val_samples = train_test_split(
    train_all, test_size=VAL_SPLIT, random_state=SEED
)

print(f"Train : {len(train_samples)}")
print(f"Val   : {len(val_samples)}")
print(f"Test  : {len(test_samples)}")

# ============================================================
# DATASET
# ============================================================

class IIIT5KDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.augment = augment

        # Base transforms (always applied)
        base = [
            T.Grayscale(1),
            T.Resize((IMG_H, IMG_W)),
            T.ToTensor(),
            T.Normalize([0.5], [0.5]),
        ]

        # Augmentation transforms (train only)
        aug = [
            T.Grayscale(1),
            T.Resize((IMG_H, IMG_W)),
            T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.3),
            T.RandomApply([
                T.RandomAffine(
                    degrees=3,
                    translate=(0.03, 0.03),
                    scale=(0.9, 1.1),
                    shear=3,
                )
            ], p=0.6),
            T.ColorJitter(brightness=0.3, contrast=0.3),
            T.ToTensor(),
            T.Normalize([0.5], [0.5]),
        ]

        self.transform = T.Compose(aug if augment else base)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        image  = Image.open(img_path).convert("RGB")
        image  = self.transform(image)
        label  = encode_label(text)
        return image, label, text

def collate_fn(batch):
    """
    CTC requires variable-length labels → pack with torch.nn.utils.rnn.pad_sequence
    Returns:
        images      : [B, 1, H, W]
        labels      : 1D concatenated tensor for CTCLoss
        label_lens  : [B] lengths of each label
        texts       : list of ground truth strings
    """
    images, labels, texts = zip(*batch)
    images     = torch.stack(images, 0)
    label_lens = torch.tensor([len(l) for l in labels], dtype=torch.long)
    labels_cat = torch.tensor([idx for l in labels for idx in l], dtype=torch.long)
    return images, labels_cat, label_lens, list(texts)

train_loader = DataLoader(
    IIIT5KDataset(train_samples, augment=True),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    IIIT5KDataset(val_samples, augment=False),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    IIIT5KDataset(test_samples, augment=False),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    collate_fn=collate_fn,
)

# ============================================================
# MODEL — CRNN
# ============================================================
#
# Pipeline:
#   Image [B,1,32,128]
#     ↓  CNN Backbone (7 conv layers, 3 max-pools)
#   Feature Map [B, 512, 1, W']     (W' = 32 after pools)
#     ↓  Squeeze height → sequence
#   Sequence [B, W', 512]
#     ↓  2-layer BiLSTM
#   Context  [B, W', 512]
#     ↓  Linear projection
#   Logits   [W', B, NUM_CLASSES]   (T-first for CTCLoss)
#     ↓  CTC decode (greedy)
#   Predicted string
#
# Why this works on small data:
#   - CNN has only ~3M params (vs 88M Swin-Base)
#   - CTC loss needs no alignment supervision
#   - BiLSTM captures context across the width sequence
# ============================================================

class CRNN(nn.Module):

    def __init__(self, num_classes, rnn_hidden=RNN_HIDDEN, rnn_layers=RNN_LAYERS):
        super().__init__()

        # ---- CNN backbone ----
        # Progressively doubles channels: 1→64→128→256→256→512→512→512
        # Height is pooled down to 1 by the end (32 → 16 → 8 → 4 → 1 via AdaptivePool)
        # Width is preserved so each column becomes a time step
        self.cnn = nn.Sequential(
            # Block 1 — 32×128 → 16×64
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 2 — 16×64 → 8×32
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 3 — 8×32 → 4×32  (pool only height)
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1), (2, 1)),      # height halved, width kept

            # Block 4 — 4×32
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),

            # Block 5 — 4×32 → 1×32
            nn.Conv2d(512, 512, 2, padding=0), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
        )

        # Collapse remaining height to 1 (handles slight size differences)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, None))

        # ---- Dropout before RNN ----
        self.dropout = nn.Dropout(0.3)

        # ---- BiLSTM sequence modelling ----
        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            bidirectional=True,
            batch_first=True,
            dropout=0.3 if rnn_layers > 1 else 0.0,
        )

        # ---- Output projection ----
        self.fc = nn.Linear(rnn_hidden * 2, num_classes)

    def forward(self, x):
        # x: [B, 1, H, W]
        feat = self.cnn(x)                        # [B, 512, ~1, W']
        feat = self.adaptive_pool(feat)           # [B, 512, 1, W']
        feat = feat.squeeze(2)                    # [B, 512, W']
        feat = feat.permute(0, 2, 1)              # [B, W', 512]
        feat = self.dropout(feat)
        out, _ = self.rnn(feat)                   # [B, W', 2*hidden]
        logits = self.fc(out)                     # [B, W', num_classes]
        logits = logits.permute(1, 0, 2)          # [W', B, num_classes] ← CTCLoss expects T-first
        return logits

# ============================================================
# INSTANTIATE
# ============================================================

model = CRNN(num_classes=NUM_CLASSES).to(device)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Total parameters: {total_params:.2f}M")

# ============================================================
# LOSS, OPTIMIZER, SCHEDULER
# ============================================================

ctc_loss = nn.CTCLoss(blank=BLANK_IDX, reduction="mean", zero_infinity=True)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

# Cosine annealing with warm restarts — good for small datasets
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=30, T_mult=2, eta_min=1e-6
)

scaler = GradScaler(enabled=USE_AMP)

# ============================================================
# METRICS
# ============================================================

def compute_metrics(pred_texts, gt_texts):
    """Returns word accuracy and character accuracy."""
    word_correct = sum(p == g for p, g in zip(pred_texts, gt_texts))
    word_acc = word_correct / len(gt_texts)

    total_chars = sum(len(g) for g in gt_texts)
    total_edits = sum(levenshtein_distance(p, g) for p, g in zip(pred_texts, gt_texts))
    char_acc = max(0.0, 1.0 - total_edits / max(total_chars, 1))

    return word_acc, char_acc

def run_inference(loader):
    """Run greedy CTC decoding on a dataloader, return (pred_texts, gt_texts)."""
    model.eval()
    all_preds = []
    all_gts   = []
    with torch.no_grad():
        for images, _, _, texts in loader:
            images = images.to(device)
            with autocast(enabled=USE_AMP):
                logits = model(images)                   # [T, B, C]
            pred_indices = logits.argmax(dim=2)          # [T, B]
            pred_indices = pred_indices.permute(1, 0)    # [B, T]
            for seq, gt in zip(pred_indices, texts):
                pred = decode_ctc(seq.cpu().tolist())
                all_preds.append(pred)
                all_gts.append(gt)
    return all_preds, all_gts

# ============================================================
# TRAINING LOOP
# ============================================================

best_val_wacc  = 0.0
patience_count = 0
history = {"train_loss": [], "val_wacc": [], "val_cacc": []}

for epoch in range(1, NUM_EPOCHS + 1):

    # ---- Train ----
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")

    for images, labels_cat, label_lens, _ in pbar:
        images     = images.to(device)
        labels_cat = labels_cat.to(device)
        label_lens = label_lens.to(device)

        optimizer.zero_grad()

        with autocast(enabled=USE_AMP):
            logits = model(images)                          # [T, B, C]
            T_len  = logits.size(0)
            B      = images.size(0)
            input_lens = torch.full((B,), T_len, dtype=torch.long, device=device)

            # CTC loss expects log-softmax input
            log_probs = F.log_softmax(logits, dim=2)
            loss = ctc_loss(log_probs, labels_cat, input_lens, label_lens)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    scheduler.step()

    train_loss = running_loss / len(train_loader)

    # ---- Validate ----
    val_preds, val_gts = run_inference(val_loader)
    val_wacc, val_cacc = compute_metrics(val_preds, val_gts)

    history["train_loss"].append(train_loss)
    history["val_wacc"].append(val_wacc)
    history["val_cacc"].append(val_cacc)

    print(f"\nEpoch {epoch:3d} | Train Loss: {train_loss:.4f} | "
          f"Val WAcc: {val_wacc*100:.2f}% | Val CAcc: {val_cacc*100:.2f}%")

    # ---- Save last ----
    torch.save(model.state_dict(), os.path.join(SAVE_DIR, "last_crnn_ctc.pth"))

    # ---- Save best ----
    if val_wacc > best_val_wacc:
        best_val_wacc  = val_wacc
        patience_count = 0
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_crnn_ctc.pth"))
        print("  ✓ Best model saved.")
    else:
        patience_count += 1
        print(f"  Early stop counter: {patience_count}/{PATIENCE}")
        if patience_count >= PATIENCE:
            print(f"\n[Early Stop] Triggered at epoch {epoch}.")
            break

print(f"\nTraining done. Best Val WAcc: {best_val_wacc*100:.2f}%")

# ============================================================
# TEST EVALUATION (load best checkpoint)
# ============================================================

print("\nLoading best model for test evaluation...")
model.load_state_dict(
    torch.load(os.path.join(SAVE_DIR, "best_crnn_ctc.pth"), map_location=device)
)

test_preds, test_gts = run_inference(test_loader)
test_wacc, test_cacc = compute_metrics(test_preds, test_gts)

print("\n" + "="*50)
print("OFFICIAL TEST RESULTS")
print("="*50)
print(f"Test Word Accuracy : {test_wacc*100:.2f}%")
print(f"Test Char Accuracy : {test_cacc*100:.2f}%")
print("="*50)

# ============================================================
# SAMPLE PREDICTIONS
# ============================================================

print("\nSAMPLE PREDICTIONS (first 15)\n")
for i in range(min(15, len(test_preds))):
    match = "✓" if test_preds[i] == test_gts[i] else "✗"
    print(f"  [{match}] GT: {test_gts[i]:<20} PRED: {test_preds[i]}")

Device : cuda
GPU    : NVIDIA RTX 5000 Ada Generation
VRAM   : 33.8 GB
Vocab size (with blank): 63


/tmp/ipykernel_741007/123062675.py:347: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=USE_AMP)


Train : 1800
Val   : 200
Test  : 3000
Total parameters: 8.74M


Epoch 1/150:   0%|                                                                                                                      | 0/113 [00:00<?, ?it/s]/tmp/ipykernel_741007/123062675.py:404: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):
Epoch 1/150: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 113/113 [00:02<00:00, 49.21it/s, loss=3.4367]
/tmp/ipykernel_741007/123062675.py:372: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):



Epoch   1 | Train Loss: 5.9649 | Val WAcc: 0.00% | Val CAcc: 0.00%
  Early stop counter: 1/30


...


Epoch 148 | Train Loss: 0.0274 | Val WAcc: 74.00% | Val CAcc: 91.61%
  Early stop counter: 11/30


Epoch 149/150: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 113/113 [00:02<00:00, 52.28it/s, loss=0.0083]



Epoch 149 | Train Loss: 0.0181 | Val WAcc: 78.00% | Val CAcc: 92.50%
  ✓ Best model saved.


Epoch 150/150: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 113/113 [00:02<00:00, 54.63it/s, loss=0.0202]



Epoch 150 | Train Loss: 0.0106 | Val WAcc: 78.00% | Val CAcc: 92.69%
  Early stop counter: 1/30

Training done. Best Val WAcc: 78.00%

Loading best model for test evaluation...

OFFICIAL TEST RESULTS
Test Word Accuracy : 63.83%
Test Char Accuracy : 82.30%

SAMPLE PREDICTIONS (first 15)

  [✓] GT: PRIVATE              PRED: PRIVATE
  [✓] GT: PARKING              PRED: PARKING
  [✓] GT: SALUTES              PRED: SALUTES
  [✓] GT: DOLCE                PRED: DOLCE
  [✗] GT: GABBANA              PRED: CHBBANA
  [✗] GT: REGENCY              PRED: MGENCI
  [✓] GT: STATE                PRED: STATE
  [✓] GT: BANK                 PRED: BANK
  [✓] GT: OF                   PRED: OF
  [✓] GT: INDIA                PRED: INDIA
  [✗] GT: KINGFISHER           PRED: KKNGFISHLER
  [✓] GT: CLEAR                PRED: CLEAR
  [✓] GT: CHANNEL              PRED: CHANNEL
  [✓] GT: UNIVERSAL            PRED: UNIVERSAL
  [✓] GT: STUDIOS              PRED: STUDIOS


##
[Raw Image] ──► [Aspect-Ratio Resize] ──► [White Padding] ──► [Swin Encoder (LR: 1e-5)] ──► [Positional Bridge] ──► [Transformer Decoder (4 Layers / LR: 5e-5)] ◄── [Native Char Embeddings] ──► [Clean Text Output]

In [5]:
# ============================================================
# HVLT FOR IIIT5k WORD RECOGNITION (OPTIMIZED FIXED VERSION)
# FULL SINGLE-CELL TRAINING & EVALUATION NOTEBOOK
# ============================================================

import os
import cv2
import time
import random
import unicodedata
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.amp import autocast, GradScaler

import timm
from jiwer import wer
from scipy.io import loadmat

# ============================================================
# CONFIG
# ============================================================
ROOT_DIR = "/home/mca/Shweta/handwritten text detection/dataset/IIIT5K-Word_V3.0 (1)/IIIT5K"

TRAIN_MAT = os.path.join(ROOT_DIR, "traindata.mat")
TEST_MAT  = os.path.join(ROOT_DIR, "testdata.mat")

IMG_HEIGHT = 32
IMG_WIDTH = 192
BATCH_SIZE = 32
MAX_EPOCHS = 20
MAX_SEQ_LEN = 40

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4
SEED = 42
USE_AMP = True

# ============================================================
# SEED SETUP
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# ENGLISH VOCABULARY (IIIT5K)
# ============================================================
CHARS = (
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
    ".,!?()[]{}:;-_'/\\&%$#@+*= "
)

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
ALL_TOKENS = SPECIAL_TOKENS + list(CHARS)

char2idx = {c: i for i, c in enumerate(ALL_TOKENS)}
idx2char = {i: c for c, i in char2idx.items()}
VOCAB_SIZE = len(ALL_TOKENS)

PAD_IDX = char2idx["<PAD>"]
SOS_IDX = char2idx["<SOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

print("VOCAB SIZE:", VOCAB_SIZE)

# ============================================================
# DATA PARSING UTILS
# ============================================================
def extract_string(x):
    while isinstance(x, np.ndarray):
        if x.size == 0:
            return ""
        x = x[0]
    return str(x)

def load_iiit5k_mat(mat_path):
    extracted_samples = []
    mat_data = loadmat(mat_path)
    # Target key deduction dynamically based on mat structure file name
    key = "traindata" if "train" in os.path.basename(mat_path) else "testdata"
    records = mat_data[key][0]
    
    for record in records:
        try:
            img_rel = extract_string(record[0])
            text = extract_string(record[1])
            img_path = os.path.join(ROOT_DIR, img_rel)
            
            if os.path.exists(img_path):
                extracted_samples.append((
                    img_path,
                    unicodedata.normalize("NFC", text.strip())
                ))
        except Exception:
            continue
    return extracted_samples

print("Loading MAT metadata files...")
samples = load_iiit5k_mat(TRAIN_MAT)
test_samples = load_iiit5k_mat(TEST_MAT)

# Split Train -> Train (90%) + Val (10%)
train_samples, val_samples = train_test_split(samples, test_size=0.10, random_state=SEED)

print(f"Dataset Split Sizes Summary -> TRAIN: {len(train_samples)} | VAL: {len(val_samples)} | TEST: {len(test_samples)}")

# ============================================================
# IMAGE PREPROCESSING ENGINE (FIXED STRETCH DISTORTION)
# ============================================================
def preprocess_image(img_path, augment=False):
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f"Cannot read image: {img_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Mild online data augmentation to minimize train overfitting
    if augment and random.random() < 0.4:
        if random.random() < 0.5:
            img = cv2.GaussianBlur(img, (3, 3), 0)
        else:
            alpha = random.uniform(0.8, 1.2)
            img = cv2.convertScaleAbs(img, alpha=alpha, beta=0)

    h, w = img.shape[:2]
    
    # Calculate aspect ratio scaling width parameters
    scale = IMG_HEIGHT / h
    new_w = max(1, int(w * scale))
    
    if new_w > IMG_WIDTH:
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    else:
        img = cv2.resize(img, (new_w, IMG_HEIGHT))
        # Pad to the right with clean white pixels instead of stretching text geometric rules
        pad_width = IMG_WIDTH - new_w
        pad = np.ones((IMG_HEIGHT, pad_width, 3), dtype=np.uint8) * 255
        img = np.concatenate([img, pad], axis=1)

    # Standard scale tensor normalization
    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    img = np.transpose(img, (2, 0, 1))
    
    return torch.tensor(img, dtype=torch.float32)

# ============================================================
# TEXT TRANSFORMATION TOKENS UTILS
# ============================================================
def encode_text(text):
    tokens = [SOS_IDX]
    for ch in text:
        tokens.append(char2idx.get(ch, UNK_IDX))
    tokens.append(EOS_IDX)

    if len(tokens) < MAX_SEQ_LEN:
        tokens += [PAD_IDX] * (MAX_SEQ_LEN - len(tokens))
    else:
        tokens = tokens[:MAX_SEQ_LEN]
        tokens[-1] = EOS_IDX
    return torch.tensor(tokens, dtype=torch.long)

def decode_tokens(tokens):
    chars = []
    for t in tokens:
        t = int(t)
        if t == EOS_IDX:
            break
        if t in [PAD_IDX, SOS_IDX]:
            continue
        chars.append(idx2char.get(t, ""))
    return "".join(chars)

# ============================================================
# PYTORCH DATASETS AND LOADERS
# ============================================================
class IIIT5KWordDataset(Dataset):
    def __init__(self, sample_list, augment=False):
        self.samples = sample_list
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        image = preprocess_image(img_path, augment=self.augment)
        tokens = encode_text(text)
        return image, tokens, text

train_loader = DataLoader(IIIT5KWordDataset(train_samples, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(IIIT5KWordDataset(val_samples, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(IIIT5KWordDataset(test_samples, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# ============================================================
# POSITIONAL BRIDGE
# ============================================================
class PositionalBridge(nn.Module):
    def __init__(self, in_channels=1024, d_model=768, vis_seq_len=256):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d((1, vis_seq_len))
        self.proj = nn.Linear(in_channels, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, vis_seq_len, d_model) * 0.02)

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = self.pool(x).squeeze(2).permute(0, 2, 1)
        return self.proj(x) + self.pos_embed

# ============================================================
# DECODER ENGINE (REMOVED BROKEN MULTILINGUAL EMBEDDINGS)
# ============================================================
class IIIT5KDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=768, n_heads=12, n_layers=4):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_embed = nn.Embedding(MAX_SEQ_LEN, d_model)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=3072, dropout=0.2, batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)
        print("Initialized localized Character Embedding Matrix layers cleanly.")

    def forward(self, memory, tgt_tokens):
        B, T = tgt_tokens.shape
        pos = torch.arange(T, device=tgt_tokens.device).unsqueeze(0).expand(B, -1)
        x = self.token_embed(tgt_tokens) + self.pos_embed(pos)

        tgt_mask = torch.triu(torch.ones(T, T, device=tgt_tokens.device), diagonal=1).bool()
        tgt_key_padding_mask = (tgt_tokens == PAD_IDX)

        out = self.decoder(tgt=x, memory=memory, tgt_mask=tgt_mask, tgt_key_padding_mask=tgt_key_padding_mask)
        return self.output_proj(out)

    @torch.no_grad()
    def greedy_decode(self, memory, max_len=MAX_SEQ_LEN):
        B = memory.shape[0]
        generated = torch.full((B, 1), SOS_IDX, device=memory.device, dtype=torch.long)
        finished = torch.zeros(B, dtype=torch.bool, device=memory.device)

        for _ in range(max_len):
            logits = self.forward(memory, generated)
            next_token = logits[:, -1].argmax(dim=-1)
            next_token = next_token.masked_fill(finished, PAD_IDX)

            generated = torch.cat([generated, next_token.unsqueeze(1)], dim=1)
            finished |= (next_token == EOS_IDX)
            if finished.all():
                break
        return generated[:, 1:]

# ============================================================
# CORE HVLT WRAPPER
# ============================================================
class HVLT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model(
            "swin_base_patch4_window7_224", pretrained=True, features_only=True,
            img_size=(32, 192), strict_img_size=False
        )
        self.bridge = PositionalBridge(in_channels=1024, d_model=768, vis_seq_len=256)
        self.decoder = IIIT5KDecoder(vocab_size=VOCAB_SIZE)

    def forward(self, images, tgt_tokens):
        feats = self.encoder(images)[-1]
        memory = self.bridge(feats)
        return self.decoder(memory, tgt_tokens)

    @torch.no_grad()
    def predict(self, images):
        feats = self.encoder(images)[-1]
        memory = self.bridge(feats)
        return self.decoder.greedy_decode(memory)

# ============================================================
# SYSTEM OPTIMIZER & CRITERIA SETUP (DIFFERENTIAL LEARNING RATES)
# ============================================================
model = HVLT().to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# Lower the encoder learning rate significantly to lock baseline ImageNet features
optimizer = Adam([
    {"params": model.encoder.parameters(), "lr": 1e-5},
    {"params": model.bridge.parameters(), "lr": 5e-5},
    {"params": model.decoder.parameters(), "lr": 5e-5}
])

scaler = GradScaler("cuda", enabled=USE_AMP)
print("TOTAL PARAMS STACK:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

def char_accuracy(preds, labels):
    correct, total = 0, 0
    for p, l in zip(preds, labels):
        n = min(len(p), len(l))
        for i in range(n):
            if p[i] == l[i]: correct += 1
        total += max(len(p), len(l))
    return 100.0 * correct / max(total, 1)

# ============================================================
# TRAINING SYSTEM LOOP
# ============================================================
best_wer = float('inf')

for epoch in range(1, MAX_EPOCHS + 1):
    # --- TRAINING STAGE ---
    model.train()
    train_loss = 0
    train_preds, train_labels = [], []
    
    pbar = tqdm(train_loader)
    for images, targets, labels in pbar:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        decoder_input = targets[:, :-1]
        decoder_target = targets[:, 1:]

        optimizer.zero_grad()
        with autocast("cuda", enabled=USE_AMP):
            logits = model(images, decoder_input)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        with torch.no_grad():
            pred_ids = logits.argmax(dim=-1)
            preds = [decode_tokens(x) for x in pred_ids]

        train_preds.extend(preds)
        train_labels.extend(labels)
        pbar.set_description(f"Epoch {epoch} | Loss: {loss.item():.4f}")

    train_cer = char_accuracy(train_preds, train_labels)
    train_wer = wer(train_labels, train_preds) * 100

    # --- VALIDATION STAGE ---
    model.eval()
    val_loss = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for images, targets, labels in val_loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            decoder_input = targets[:, :-1]
            decoder_target = targets[:, 1:]

            logits = model(images, decoder_input)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1))
            val_loss += loss.item()

            pred_ids = model.predict(images)
            preds = [decode_tokens(x) for x in pred_ids]
            val_preds.extend(preds)
            val_labels.extend(labels)

    val_cer = char_accuracy(val_preds, val_labels)
    val_wer = wer(val_labels, val_preds) * 100

    print("\n" + "="*60)
    print(f"EPOCH {epoch}")
    print(f"TRAIN LOSS: {train_loss / len(train_loader):.4f} | VAL LOSS: {val_loss / len(val_loader):.4f}")
    print(f"TRAIN CAR:  {train_cer:.2f}%  | VAL CAR:  {val_cer:.2f}%")
    print(f"TRAIN WER:  {train_wer:.2f}%  | VAL WER:  {val_wer:.2f}%")
    print("="*60)

    if val_wer < best_wer:
        best_wer = val_wer
        torch.save({"model": model.state_dict(), "epoch": epoch, "val_wer": val_wer}, "best_iiit5k_hvlt.pth")
        print(">>> CRITERIA METRICS MATCHED: SAVE NEW OPTIMAL WEIGHTS CHECKPOINT.")

# ============================================================
# FINAL DATA COMPUTE EVALUATION RUN
# ============================================================
print("\nLOADING OPTIMIZED CHECKPOINT MARKS FOR FINAL SYSTEM ASSESSMENT...")
if os.path.exists("best_iiit5k_hvlt.pth"):
    checkpoint = torch.load("best_iiit5k_hvlt.pth")
    model.load_state_dict(checkpoint["model"])

model.eval()
test_preds, test_labels = [], []

with torch.no_grad():
    for images, _, labels in tqdm(test_loader, desc="Testing Assessment"):
        images = images.to(DEVICE)
        pred_ids = model.predict(images)
        preds = [decode_tokens(x) for x in pred_ids]
        test_preds.extend(preds)
        test_labels.extend(labels)

test_cer = char_accuracy(test_preds, test_labels)
test_wer = wer(test_labels, test_preds) * 100

print("\n" + "#"*40 + "\nFINAL TEST SYSTEM GENERALIZATION MATRIX")
print(f"FINAL TEST CAR : {test_cer:.2f}%")
print(f"FINAL TEST WER : {test_wer:.2f}%")
print("#"*40)

# Preview visual inspection verification paths
print("\nSAMPLE VISUAL TESTS PREVIEW ANALYSES:\n")
for i in range(10):
    print(f"[{i+1:02d}] GROUND TRUTH : {test_labels[i]}")
    print(f"     MODEL PREDICT : {test_preds[i]}")
    print("-" * 50)

VOCAB SIZE: 92
Loading MAT metadata files...
Dataset Split Sizes Summary -> TRAIN: 1800 | VAL: 200 | TEST: 3000


/home/mca/Shweta/handwritten text detection/sbenv/lib/python3.12/site-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at /pytorch/aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)
/home/mca/Shweta/handwritten text detection/sbenv/lib/python3.12/site-packages/timm/layers/interpolate.py:65: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:34

Initialized localized Character Embedding Matrix layers cleanly.
TOTAL PARAMS STACK: 125.655412 M


Epoch 1 | Loss: 2.5864: 100%|████████████████████████████████████████████████████████████████████| 57/57 [00:07<00:00,  7.54it/s]



EPOCH 1
TRAIN LOSS: 3.1842 | VAL LOSS: 2.9253
TRAIN CAR:  2.99%  | VAL CAR:  6.83%
TRAIN WER:  100.00%  | VAL WER:  100.00%
>>> CRITERIA METRICS MATCHED: SAVE NEW OPTIMAL WEIGHTS CHECKPOINT.


...


EPOCH 19
TRAIN LOSS: 0.4774 | VAL LOSS: 2.3654
TRAIN CAR:  82.74%  | VAL CAR:  21.24%
TRAIN WER:  49.06%  | VAL WER:  89.50%


Epoch 20 | Loss: 0.4842: 100%|███████████████████████████████████████████████████████████████████| 57/57 [00:07<00:00,  8.01it/s]



EPOCH 20
TRAIN LOSS: 0.4204 | VAL LOSS: 2.3454
TRAIN CAR:  84.05%  | VAL CAR:  22.90%
TRAIN WER:  44.72%  | VAL WER:  88.50%

LOADING OPTIMIZED CHECKPOINT MARKS FOR FINAL SYSTEM ASSESSMENT...


Testing Assessment: 100%|████████████████████████████████████████████████████████████████████████| 94/94 [00:14<00:00,  6.70it/s]



########################################
FINAL TEST SYSTEM GENERALIZATION MATRIX
FINAL TEST CAR : 19.25%
FINAL TEST WER : 88.83%
########################################

SAMPLE VISUAL TESTS PREVIEW ANALYSES:

[01] GROUND TRUTH : PRIVATE
     MODEL PREDICT : MADEYS
--------------------------------------------------
[02] GROUND TRUTH : PARKING
     MODEL PREDICT : ALLADAY
--------------------------------------------------
[03] GROUND TRUTH : SALUTES
     MODEL PREDICT : SRIVERS
--------------------------------------------------
[04] GROUND TRUTH : DOLCE
     MODEL PREDICT : DODDY
--------------------------------------------------
[05] GROUND TRUTH : GABBANA
     MODEL PREDICT : SANGAM
--------------------------------------------------
[06] GROUND TRUTH : REGENCY
     MODEL PREDICT : GELLAGARD
--------------------------------------------------
[07] GROUND TRUTH : STATE
     MODEL PREDICT : FAST
--------------------------------------------------
[08] GROUND TRUTH : BANK
     MODEL PREDIC